<a href="https://colab.research.google.com/github/manas20o5/UNLOX-ASSIMENTS/blob/main/Minor_project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:

!pip install numpy

import numpy as np
from datetime import datetime, timedelta

print("GroupDNA is ready!")
print("NumPy version:", np.__version__)


GroupDNA is ready!
NumPy version: 2.4.6


In [9]:

file_path = "/content/hostel_bois.txt"

with open(file_path, "r", encoding="utf-8") as file:
    chat_data = file.read()

print("Dataset uploaded successfully!")
print("Total characters:", len(chat_data))
print("\nFirst 500 characters:\n")
print(chat_data[:500])


Dataset uploaded successfully!
Total characters: 249033

First 500 characters:

01/04/24, 01:12 - Messages and calls are end-to-end encrypted. No one outside of this chat, not even WhatsApp, can read or listen to them. Tap to learn more.
01/04/24, 01:13 - Priya created group "Hostel Bois 4ever"
01/04/24, 01:14 - Priya added you
01/04/24, 01:17 - Rahul: scene fix
01/04/24, 01:17 - Rahul: haan
01/04/24, 01:18 - Rahul: kya scene
01/04/24, 02:13 - Rahul: abhi free hai?
01/04/24, 02:13 - Rahul: abey
01/04/24, 02:14 - Rahul: bhai sun na
01/04/24, 07:46 - Priya: I have extra notes


## Feature 1: WhatsApp Chat Parser
This feature reads the WhatsApp chat file and extracts timestamps, sender names, and message text.

In [11]:

# FEATURE 1: WHATSAPP CHAT PARSER

file_path = "/content/hostel_bois.txt"

messages = []

system_count = 0
media_count = 0
deleted_count = 0


def is_timestamp_line(line):
    if len(line) < 15:
        return False

    try:
        datetime.strptime(line[:15], "%d/%m/%y, %H:%M")
        return True
    except ValueError:
        return False



with open(file_path, "r", encoding="utf-8") as file:
    lines = file.readlines()

for line in lines:
    line = line.rstrip("\n")

    if line.strip() == "":
        continue

    if is_timestamp_line(line):
        parts = line.split(" - ", 1)

        if len(parts) != 2:
            system_count += 1
            continue

        timestamp = parts[0]
        remaining = parts[1]

        if ": " not in remaining:
            system_count += 1
            continue

        sender, message = remaining.split(": ", 1)

        messages.append({
            "timestamp": timestamp,
            "sender": sender,
            "text": message
        })

    else:
        # Attach continuation lines to the previous message
        if messages:
            messages[-1]["text"] += "\n" + line


# Count media and deleted messages
for msg in messages:
    if msg["text"] == "<Media omitted>":
        media_count += 1

    elif msg["text"] == "This message was deleted":
        deleted_count += 1


print("========== CHAT PARSER RESULT ==========")
print("Total parsed entries:", len(messages))
print("System messages skipped:", system_count)
print("Media messages:", media_count)
print("Deleted messages:", deleted_count)

print("\nFirst 5 parsed messages:")
for msg in messages[:5]:
    print(msg)

print("\nLast 5 parsed messages:")
for msg in messages[-5:]:
    print(msg)


========== CHAT PARSER RESULT ==========
Total parsed entries: 3174
System messages skipped: 4
Media messages: 32
Deleted messages: 15

First 5 parsed messages:
{'timestamp': '01/04/24, 01:17', 'sender': 'Rahul', 'text': 'scene fix'}
{'timestamp': '01/04/24, 01:17', 'sender': 'Rahul', 'text': 'haan'}
{'timestamp': '01/04/24, 01:18', 'sender': 'Rahul', 'text': 'kya scene'}
{'timestamp': '01/04/24, 02:13', 'sender': 'Rahul', 'text': 'abhi free hai?'}
{'timestamp': '01/04/24, 02:13', 'sender': 'Rahul', 'text': 'abey'}

Last 5 parsed messages:
{'timestamp': '30/05/24, 19:14', 'sender': 'Priya', 'text': 'Take care everyone'}
{'timestamp': '30/05/24, 19:28', 'sender': 'Priya', 'text': 'Karan that sounds tough, take care'}
{'timestamp': '30/05/24, 21:17', 'sender': 'Aman', 'text': 'the existential dread is back'}
{'timestamp': '30/05/24, 21:30', 'sender': 'Karan', 'text': 'Long day guys, woke up at six for that placement workshop which started at eight by the way classic, then ran around camp

## Feature 2: Group Overview

This feature displays total messages, group participants, date range, and individual member statistics.

In [12]:

# FEATURE 2: GROUP OVERVIEW

# Count messages sent by each person
person_counts = {}

for msg in messages:
    sender = msg["sender"]

    if sender not in person_counts:
        person_counts[sender] = 0

    person_counts[sender] += 1


# Get unique participants
participants = set(person_counts.keys())

# Convert timestamps into datetime objects
dates = []

for msg in messages:
    date = datetime.strptime(
        msg["timestamp"], "%d/%m/%y, %H:%M"
    )
    dates.append(date)

first_date = min(dates)
last_date = max(dates)

total_days = (last_date.date() - first_date.date()).days + 1

# Calculate word count and average message length
print("=" * 55)
print("                 GROUP OVERVIEW")
print("=" * 55)

print("Group Name          : Hostel Bois 4ever")
print("Total Messages      :", len(messages))
print("Total Participants  :", len(participants))
print("Date Range          :", first_date.strftime("%d %B %Y"),
      "to", last_date.strftime("%d %B %Y"))
print("Total Days          :", total_days)

print("\nPARTICIPANT MESSAGE COUNTS")
print("-" * 55)

sorted_people = sorted(
    person_counts.items(),
    key=lambda item: item[1],
    reverse=True
)

for person, count in sorted_people:
    print(f"{person:<15} : {count} messages")

print("\nINDIVIDUAL STATISTICS")
print("-" * 55)

for person in participants:
    person_messages = [
        msg for msg in messages
        if msg["sender"] == person
        and msg["text"] not in [
            "<Media omitted>",
            "This message was deleted"
        ]
    ]

    total_words = 0
    total_characters = 0

    for msg in person_messages:
        words = msg["text"].split()
        total_words += len(words)
        total_characters += len(msg["text"])

    if len(person_messages) > 0:
        avg_length = total_characters / len(person_messages)
    else:
        avg_length = 0

    print(f"\n{person}")
    print("  Word Count:", total_words)
    print("  Average Message Length:", round(avg_length, 2), "characters")


                 GROUP OVERVIEW
Group Name          : Hostel Bois 4ever
Total Messages      : 3174
Total Participants  : 6
Date Range          : 01 April 2024 to 30 May 2024
Total Days          : 60

PARTICIPANT MESSAGE COUNTS
-------------------------------------------------------
Rahul           : 953 messages
Priya           : 718 messages
Neha            : 635 messages
Aman            : 490 messages
Karan           : 354 messages
Vikas           : 24 messages

INDIVIDUAL STATISTICS
-------------------------------------------------------

Rahul
  Word Count: 2399
  Average Message Length: 10.69 characters

Neha
  Word Count: 3317
  Average Message Length: 25.37 characters

Vikas
  Word Count: 40
  Average Message Length: 7.23 characters

Karan
  Word Count: 19681
  Average Message Length: 311.41 characters

Aman
  Word Count: 2430
  Average Message Length: 26.72 characters

Priya
  Word Count: 3560
  Average Message Length: 28.21 characters


## Feature 3: Word Frequency Analysis

This feature identifies the most frequently used words across the group and for each participant using Python dictionaries.

In [14]:

# FEATURE 3: WORD FREQUENCY ANALYSIS

# Common words to ignore
stop_words = {
    "the", "is", "are", "a", "an", "to", "of", "in",
    "on", "for", "and", "or", "it", "this", "that",
    "i", "you", "he", "she", "we", "they", "my",
    "me", "your", "our", "was", "were", "be", "am",
    "hai", "hain", "ho", "ka", "ki", "ke",
    "how", "guys", "today", "so", "about", "at",
    "his", "have", "just", "everyone", "what",
    "which", "anyone"
}
# Function to clean and count words
def count_words(message_list):
    word_count = {}

    for msg in message_list:
        if msg["text"] in ["<Media omitted>", "This message was deleted"]:
            continue

        text = msg["text"].lower()

        # Remove punctuation
        for symbol in ".,!?;:'\"()[]{}<>@#$%^&*_+-=/":
            text = text.replace(symbol, " ")

        words = text.split()

        for word in words:
            if word not in stop_words and len(word) > 1:
                if word not in word_count:
                    word_count[word] = 0

                word_count[word] += 1

    return word_count


# Group-wide word frequency
group_word_count = count_words(messages)

top_group_words = sorted(
    group_word_count.items(),
    key=lambda item: item[1],
    reverse=True
)

print("=" * 55)
print("             TOP 10 GROUP WORDS")
print("=" * 55)

for word, count in top_group_words[:10]:
    print(f"{word:<15} : {count:>5} {'#' * min(count, 30)}")


# Top 5 words for every participant
print("\n" + "=" * 55)
print("          TOP 5 WORDS PER PARTICIPANT")
print("=" * 55)

for person in sorted(participants):
    person_messages = [
        msg for msg in messages
        if msg["sender"] == person
    ]

    person_word_count = count_words(person_messages)

    top_person_words = sorted(
        person_word_count.items(),
        key=lambda item: item[1],
        reverse=True
    )

    print("\n" + person)

    for word, count in top_person_words[:5]:
        print(f"  {word:<15} : {count}")


             TOP 10 GROUP WORDS
telling         :   179 ##############################
from            :   174 ##############################
up              :   172 ##############################
bhai            :   160 ##############################
one             :   157 ##############################
had             :   151 ##############################
started         :   150 ##############################
no              :   146 ##############################
scene           :   145 ##############################
entire          :   145 ##############################

          TOP 5 WORDS PER PARTICIPANT

Aman
  sleep           : 71
  wonder          : 43
  night           : 41
  can             : 34
  does            : 33

Karan
  telling         : 179
  up              : 164
  from            : 154
  started         : 150
  entire          : 145

Neha
  cant            : 58
  ok              : 52
  no              : 48
  way             : 48
  wait            : 48

Priya
  p

## Feature 4: NumPy Activity Heatmap

This feature uses NumPy to calculate the number of messages sent by each participant during every hour of the day.

In [15]:

# FEATURE 4: NUMPY ACTIVITY HEATMAP

people = ["Rahul", "Priya", "Aman", "Karan", "Neha", "Vikas"]

# Create a 6 x 24 matrix filled with zeros
activity_matrix = np.zeros((6, 24), dtype=int)

# Count messages for each person and hour
for msg in messages:
    sender = msg["sender"]

    if sender in people:
        person_index = people.index(sender)

        date = datetime.strptime(
            msg["timestamp"], "%d/%m/%y, %H:%M"
        )

        hour = date.hour

        activity_matrix[person_index][hour] += 1


# Display the heatmap
print("=" * 75)
print("                    GROUP ACTIVITY HEATMAP")
print("=" * 75)

print("Person".ljust(10), end="")

for hour in range(24):
    print(f"{hour:02}", end=" ")

print("\n" + "-" * 75)

# Four levels of shading
symbols = [" ", "░", "▒", "▓", "█"]

max_value = np.max(activity_matrix)

for i in range(len(people)):
    print(people[i].ljust(10), end="")

    for hour in range(24):
        value = activity_matrix[i][hour]

        if value == 0:
            symbol = " "
        elif value <= max_value * 0.25:
            symbol = "░"
        elif value <= max_value * 0.50:
            symbol = "▒"
        elif value <= max_value * 0.75:
            symbol = "▓"
        else:
            symbol = "█"

        print(symbol, end="  ")

    print()

print("\nLegend: ░ Low | ▒ Medium | ▓ High | █ Very High")

print("\nTotal messages in heatmap:", np.sum(activity_matrix))
print("Matrix shape:", activity_matrix.shape)


                    GROUP ACTIVITY HEATMAP
Person    00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23 
---------------------------------------------------------------------------
Rahul     ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ▓  ▒  ▒  ▓  ▓  ▒  █  ▓  ▒  █  ▓  ▓  
Priya                       ░  ░  ▒  ▓  ▓  ▓  ▓  ▒  ▒  ▒  ▒  ▒  ▒  ▓  ▒  ▒  ░  ░  
Aman      ▓  ▓  ▓  ▓  █                             ░  ░  ░  ░  ░  ░  ░  ░     ▓  
Karan                          ░  ░  ░  ░  ░  ▒  ░  ▒  ▒  ▒  ▒  ░  ▒  ░  ░  ░  ░  
Neha                     ░  ░  ░  ▒  ▒  ▒  ░  ▒  ▒  ▒  ░  ▒  ▒  ▓  ▒  ▒  ▒  ▒  ▒  
Vikas                          ░  ░  ░  ░     ░  ░     ░  ░  ░  ░  ░  ░  ░  ░  ░  

Legend: ░ Low | ▒ Medium | ▓ High | █ Very High

Total messages in heatmap: 3174
Matrix shape: (6, 24)


## Feature 5: Response Time Analysis

This feature calculates the average time a participant takes to respond after another participant sends a message.

In [16]:

# FEATURE 5: RESPONSE TIME ANALYSIS

# Convert messages into datetime format
dated_messages = []

for msg in messages:
    date = datetime.strptime(
        msg["timestamp"], "%d/%m/%y, %H:%M"
    )

    dated_messages.append({
        "timestamp": date,
        "sender": msg["sender"],
        "text": msg["text"]
    })

# Sort messages by timestamp
dated_messages.sort(key=lambda x: x["timestamp"])

# Store response times for each participant
response_times = {}

for person in participants:
    response_times[person] = []

# Compare consecutive messages
for i in range(1, len(dated_messages)):
    previous = dated_messages[i - 1]
    current = dated_messages[i]

    if previous["sender"] != current["sender"]:
        gap = (
            current["timestamp"] - previous["timestamp"]
        ).total_seconds() / 60

        response_times[current["sender"]].append(gap)

# Display average response times
print("=" * 60)
print("             AVERAGE RESPONSE TIMES")
print("=" * 60)

for person in sorted(participants):
    times = response_times[person]

    if len(times) > 0:
        average = sum(times) / len(times)
        print(f"{person:<15}: {average:.2f} minutes")
    else:
        print(f"{person:<15}: No response data")


             AVERAGE RESPONSE TIMES
Aman           : 55.36 minutes
Karan          : 36.62 minutes
Neha           : 39.45 minutes
Priya          : 41.99 minutes
Rahul          : 34.95 minutes
Vikas          : 46.30 minutes


## Feature 6: Silent Streak Analysis

This feature calculates the longest period of inactivity for every participant.

In [20]:

# FEATURE 6: SILENT STREAKS

person_active_dates = {}

for person in participants:
    person_active_dates[person] = set()

for msg in dated_messages:
    person_active_dates[msg["sender"]].add(
        msg["timestamp"].date()
    )

silent_streaks = {}

for person in sorted(participants):
    active_dates = sorted(person_active_dates[person])

    longest_streak = 0

    if not active_dates:
        longest_streak = total_days
    else:
        # Silence before the first active date
        start_gap = (active_dates[0] - first_date.date()).days
        longest_streak = max(longest_streak, start_gap)

        # Silence between active dates
        for i in range(1, len(active_dates)):
            gap = (active_dates[i] - active_dates[i - 1]).days - 1
            longest_streak = max(longest_streak, gap)

        # Silence after the last active date
        end_gap = (last_date.date() - active_dates[-1]).days
        longest_streak = max(longest_streak, end_gap)

    silent_streaks[person] = longest_streak

print("CORRECTED SILENT STREAKS")

for person, streak in sorted(
    silent_streaks.items(),
    key=lambda item: item[1],
    reverse=True
):
    print(person, ":", streak, "days")


CORRECTED SILENT STREAKS
Vikas : 11 days
Aman : 0 days
Karan : 0 days
Neha : 0 days
Priya : 0 days
Rahul : 0 days


## Feature 7: Personality Archetype Detection

This feature analyzes participant messaging patterns and assigns one personality archetype to each participant based on numerical scores.

In [18]:

# FEATURE 7: PERSONALITY ARCHETYPE DETECTION

# Caring and friendly keywords
caring_words = [
    "okay", "safe", "eat", "sleep", "take care",
    "are you", "please", "reminder", "drink water",
    "don't forget"
]

funny_words = ["lol", "lmao", "haha", "rofl", "lmfao"]

archetype_results = {}

for person in sorted(participants):

    person_msgs = [
        msg for msg in messages
        if msg["sender"] == person
    ]

    total = len(person_msgs)

    # Exclude media and deleted messages from text analysis
    valid_msgs = [
        msg["text"] for msg in person_msgs
        if msg["text"] not in [
            "<Media omitted>",
            "This message was deleted"
        ]
    ]

    # 1. SPAMMER: Average consecutive message burst
    bursts = []
    current_burst = 0

    for msg in messages:
        if msg["sender"] == person:
            current_burst += 1
        else:
            if current_burst > 0:
                bursts.append(current_burst)
            current_burst = 0

    if current_burst > 0:
        bursts.append(current_burst)

    avg_burst = (
        sum(bursts) / len(bursts) if bursts else 0
    )

    # 2. GROUP MOM: Count caring keywords
    caring_count = 0

    for msg in valid_msgs:
        lower_msg = msg.lower()

        for word in caring_words:
            caring_count += lower_msg.count(word)

    # 3. NIGHT OWL: Messages between 23:00 and 04:59
    night_count = 0

    for msg in person_msgs:
        hour = datetime.strptime(
            msg["timestamp"], "%d/%m/%y, %H:%M"
        ).hour

        if hour >= 23 or hour <= 4:
            night_count += 1

    night_percentage = (
        night_count / total * 100 if total else 0
    )

    # 4. STORYTELLER: Average words per message
    word_total = 0

    for msg in valid_msgs:
        word_total += len(msg.split())

    avg_words = (
        word_total / len(valid_msgs) if valid_msgs else 0
    )

    # 5. DRAMA QUEEN: Uppercase or multiple exclamation marks
    drama_count = 0

    for msg in valid_msgs:
        letters = [ch for ch in msg if ch.isalpha()]

        is_uppercase = (
            len(msg) >= 3
            and len(letters) > 0
            and all(ch.isupper() for ch in letters)
        )

        has_exclamations = msg.count("!") >= 2

        if is_uppercase or has_exclamations:
            drama_count += 1

    drama_percentage = (
        drama_count / len(valid_msgs) * 100
        if valid_msgs else 0
    )

    # 6. GHOST: Percentage of silent days
    active_days = len(person_active_dates[person])
    silent_days = total_days - active_days

    ghost_percentage = (
        silent_days / total_days * 100 if total_days else 0
    )

    # 7. COMEDIAN: Percentage of messages with funny words
    funny_count = 0

    for msg in valid_msgs:
        lower_msg = msg.lower()

        if any(word in lower_msg for word in funny_words):
            funny_count += 1

    funny_percentage = (
        funny_count / len(valid_msgs) * 100
        if valid_msgs else 0
    )

    # 8. QUESTION MASTER: Percentage ending in ?
    question_count = sum(
        1 for msg in valid_msgs if msg.strip().endswith("?")
    )

    question_percentage = (
        question_count / len(valid_msgs) * 100
        if valid_msgs else 0
    )

    # Calculate scores using the brief's suggested thresholds
    scores = {
        "THE SPAMMER": avg_burst / 3,
        "THE GROUP MOM": caring_count / 10,
        "THE NIGHT OWL": night_percentage / 60,
        "THE STORYTELLER": avg_words / 30,
        "THE DRAMA QUEEN": drama_percentage / 30,
        "THE GHOST": ghost_percentage / 60,
        "THE COMEDIAN": funny_percentage / 10,
        "THE QUESTION MASTER": question_percentage / 25
    }

    # Select exactly one highest-scoring archetype
    best_archetype = max(scores, key=scores.get)

    archetype_results[person] = {
        "archetype": best_archetype,
        "scores": scores
    }


# Display the results
print("=" * 65)
print("              PERSONALITY ARCHETYPES")
print("=" * 65)

for person, result in archetype_results.items():
    print(
        f"{person:<12} -> {result['archetype']:<25}"
        f" Score: {result['scores'][result['archetype']]:.2f}"
    )


              PERSONALITY ARCHETYPES
Aman         -> THE GROUP MOM             Score: 9.90
Karan        -> THE GROUP MOM             Score: 2.90
Neha         -> THE GROUP MOM             Score: 2.30
Priya        -> THE GROUP MOM             Score: 62.10
Rahul        -> THE SPAMMER               Score: 1.51
Vikas        -> THE COMEDIAN              Score: 1.82


## Feature 8: Final GroupDNA Report

This feature combines all the previous analysis into one formatted report showing group activity, popular words, response patterns, silent streaks, and personality archetypes.

In [19]:

# FEATURE 8: FINAL GROUPDNA REPORT

# Calculate messages per day
daily_counts = {}

for msg in dated_messages:
    day = msg["timestamp"].date()

    if day not in daily_counts:
        daily_counts[day] = 0

    daily_counts[day] += 1

busiest_day = max(daily_counts, key=daily_counts.get)

# Calculate busiest hour
hour_counts = np.sum(activity_matrix, axis=0)
busiest_hour = int(np.argmax(hour_counts))

# Calculate response-time fastest and slowest
average_responses = {}

for person, times in response_times.items():
    if times:
        average_responses[person] = sum(times) / len(times)

# Report heading
print("\n" + "=" * 75)
print('             GROUPDNA REPORT - "HOSTEL BOIS 4EVER"')
print("=" * 75)

print(
    f"{total_days} days | {len(messages):,} messages | "
    f"{len(participants)} members"
)

print("-" * 75)

print("\nGROUP SUMMARY")
print("Period          :", first_date.strftime("%d %B %Y"),
      "to", last_date.strftime("%d %B %Y"))
print("Busiest day     :", busiest_day.strftime("%d %B %Y"),
      f"({daily_counts[busiest_day]} messages)")
print(f"Busiest hour    : {busiest_hour:02d}:00 - "
      f"{(busiest_hour + 1) % 24:02d}:00")

# Messages per person
print("\nMESSAGES PER PERSON")
print("-" * 75)

for person, count in sorted(
    person_counts.items(),
    key=lambda item: item[1],
    reverse=True
):
    percentage = count / len(messages) * 100
    bar = "#" * int(percentage / 2)

    print(
        f"{person:<12} {bar:<22} "
        f"{count:>5} ({percentage:>5.1f}%)"
    )

# Favourite words
print("\nGROUP'S FAVOURITE WORDS")
print("-" * 75)

for word, count in top_group_words[:5]:
    bar = "#" * min(count // 10, 30)
    print(f"{word:<15} {bar:<30} {count}")

# Heatmap
print("\nACTIVITY HEATMAP")
print("-" * 75)

print("Person".ljust(10), end="")

for hour in range(24):
    print(f"{hour:02}", end=" ")

print()

symbols = [" ", "░", "▒", "▓", "█"]
max_value = np.max(activity_matrix)

for i, person in enumerate(people):
    print(person.ljust(10), end="")

    for hour in range(24):
        value = activity_matrix[i][hour]

        if value == 0:
            symbol = " "
        elif value <= max_value * 0.25:
            symbol = "░"
        elif value <= max_value * 0.50:
            symbol = "▒"
        elif value <= max_value * 0.75:
            symbol = "▓"
        else:
            symbol = "█"

        print(symbol, end="  ")

    print()

# Response patterns
print("\nRESPONSE PATTERNS")
print("-" * 75)

if average_responses:
    fastest = min(average_responses, key=average_responses.get)
    slowest = max(average_responses, key=average_responses.get)

    print(f"Fastest average gap : {fastest} "
          f"({average_responses[fastest]:.2f} minutes)")

    print(f"Slowest average gap : {slowest} "
          f"({average_responses[slowest]:.2f} minutes)")

# Silent streaks
print("\nLONGEST SILENT STREAKS")
print("-" * 75)

for person, streak in sorted(
    silent_streaks.items(),
    key=lambda item: item[1],
    reverse=True
):
    print(f"{person:<15}: {streak} days")

# Personality archetypes
print("\nPERSONALITY ARCHETYPES")
print("-" * 75)

for person, result in archetype_results.items():
    print(f"{person:<12} -> {result['archetype']}")

print("\n" + "=" * 75)
print("       Generated by GroupDNA | Built with Python + NumPy")
print("=" * 75)



             GROUPDNA REPORT - "HOSTEL BOIS 4EVER"
60 days | 3,174 messages | 6 members
---------------------------------------------------------------------------

GROUP SUMMARY
Period          : 01 April 2024 to 30 May 2024
Busiest day     : 04 May 2024 (76 messages)
Busiest hour    : 18:00 - 19:00

MESSAGES PER PERSON
---------------------------------------------------------------------------
Rahul        ###############          953 ( 30.0%)
Priya        ###########              718 ( 22.6%)
Neha         ##########               635 ( 20.0%)
Aman         #######                  490 ( 15.4%)
Karan        #####                    354 ( 11.2%)
Vikas                                  24 (  0.8%)

GROUP'S FAVOURITE WORDS
---------------------------------------------------------------------------
telling         #################              179
from            #################              174
up              #################              172
bhai            ################        